My pipeline (Image input after hair removal)

In [2]:
import os, cv2, torch, re, random
import numpy as np
from tqdm import tqdm
from models.UltraLight_VM_UNet import UltraLight_VM_UNet
from sklearn.metrics import confusion_matrix

# --- 1. Path Configuration ---
# Important: Ensure pointing to the fixed-seed fine-tuned weights
WEIGHT_PATH = "/gpfs/work/bio/yixuanli2204/FYP_Project/weights/vmunet/vmunet_finetuned_fixed_seed.pth"
IMG_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/restored_image_final"
MASK_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/lesion_mask"
DEVICE = torch.device("cuda")

def run_pure_test():
    # --- 2. Strict Dataset Reproduction Logic (Must match fine-tuning script) ---
    random.seed(42)
    # Get all .png images and sort to ensure initial order consistency
    all_files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.png')])
    
    # Filter out images without masks
    valid_files = []
    for f in all_files:
        match = re.search(r"ISIC_(\d{7})", f)
        img_id = match.group(1) if match else f.split('.')[0]
        if os.path.exists(os.path.join(MASK_DIR, f"ISIC_{img_id}_lesion.png")):
            valid_files.append(f)
            
    # Shuffle order
    random.shuffle(valid_files)
    
    # 【Core】: Skip first 100 images (used for fine-tuning), take only the remaining 393
    pure_test_files = valid_files[100:]
    
    print(f"Starting [Pure Test]...")
    print(f"   - Total available samples: {len(valid_files)}")
    print(f"   - Excluded training samples: 100")
    print(f"   - Actual test samples: {len(pure_test_files)}")

    # --- 3. Load fine-tuned model ---
    model = UltraLight_VM_UNet(num_classes=1, input_channels=3, c_list=[8,16,24,32,48,64], split_att='fc', bridge=True).to(DEVICE)
    checkpoint = torch.load(WEIGHT_PATH, map_location=DEVICE)
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    model.load_state_dict({k.replace('module.', ''): v for k, v in state_dict.items()})
    model.eval()

    all_metrics = []

    # --- 4. Loop inference ---
    for fname in tqdm(pure_test_files):
        match = re.search(r"ISIC_(\d{7})", fname)
        img_id = match.group(1)
        mask_path = os.path.join(MASK_DIR, f"ISIC_{img_id}_lesion.png")

        img = cv2.imread(os.path.join(IMG_DIR, fname))
        mask_gt = cv2.imread(mask_path, 0)
        
        # Preprocessing (256, 256)
        img_t = torch.from_numpy(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), (256, 256))).permute(2,0,1).unsqueeze(0).float().to(DEVICE) / 255.0
        mask_gt = (cv2.resize(mask_gt, (256, 256)) > 127).astype(np.uint8)

        with torch.no_grad():
            output = model(img_t)
            pred = (output.squeeze().cpu().numpy() > 0.5).astype(np.uint8)

        cm = confusion_matrix(mask_gt.flatten(), pred.flatten(), labels=[0, 1])
        all_metrics.append(cm)

    # --- 5. Summary calculation ---
    total_cm = sum(all_metrics)
    tn, fp, fn, tp = total_cm.ravel()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-6)
    iou = tp / (tp + fp + fn + 1e-6)
    acc = (tp + tn) / (tp + tn + fp + fn + 1e-6)
    sen = tp / (tp + fn + 1e-6)
    spe = tn / (tn + fp + 1e-6)

    
    print(f"Pure Test Report (Independent Test Set)")
    print(f"Dice (F1):    {dice:.4f}")
    print(f"Jaccard(IoU): {iou:.4f}")
    print(f"Accuracy:     {acc:.4f}")
    print(f"Sensitivity:  {sen:.4f}")
    print(f"Specificity:  {spe:.4f}")


if __name__ == "__main__":
    run_pure_test()

Starting [Pure Test]...
   - Total available samples: 493
   - Excluded training samples: 100
   - Actual test samples: 393
SC_Att_Bridge was used


100%|██████████| 393/393 [00:30<00:00, 12.79it/s]

Pure Test Report (Independent Test Set)
Dice (F1):    0.8857
Jaccard(IoU): 0.7949
Accuracy:     0.9514
Sensitivity:  0.8696
Specificity:  0.9740


Baseline (Original image input)

In [3]:
import os
import cv2
import torch
import re
import random
import numpy as np
from tqdm import tqdm
from models.UltraLight_VM_UNet import UltraLight_VM_UNet
from sklearn.metrics import confusion_matrix

# --- 1. Path Configuration for Baseline Testing ---
# Point to the 150-epoch original best weights (the foundation model)
WEIGHT_PATH = "/gpfs/work/bio/yixuanli2204/FYP_Project/code/UltraLight-VM-UNet/results/UltraLight_VM_UNet_ISIC_Combined_20260323_best/checkpoints/best.pth"
# Point to the raw hair-occluded images (Baseline input)
IMG_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/dermoscopic_image"
MASK_DIR = "/gpfs/work/bio/yixuanli2204/FYP_Project/data/gold_real_d2/lesion_mask"
DEVICE = torch.device("cuda")

def run_baseline_pure_test():
    # --- 2. Reproducible Slicing (Independent Test Set) ---
    # Use the same seed as the fine-tuning script to ensure identical sample selection
    random.seed(42)
    
    # Sort files to maintain consistent indexing
    all_files = sorted([f for f in os.listdir(IMG_DIR) if f.lower().endswith(('.png', '.jpg'))])
    
    # Filter valid pairs
    valid_files = []
    for f in all_files:
        match = re.search(r"ISIC_(\d{7})", f)
        img_id = match.group(1) if match else f.split('.')[0]
        mask_path = os.path.join(MASK_DIR, f"ISIC_{img_id}_lesion.png")
        if os.path.exists(mask_path):
            valid_files.append(f)
            
    # Shuffle and skip the first 100 images used in fine-tuning
    random.shuffle(valid_files)
    pure_test_list = valid_files[100:]
    
    print(f"Total samples: {len(valid_files)}")
    print(f"Testing on {len(pure_test_list)} independent baseline samples...")

    # --- 3. Model Loading ---
    model = UltraLight_VM_UNet(num_classes=1, input_channels=3, c_list=[8,16,24,32,48,64], split_att='fc', bridge=True).to(DEVICE)
    checkpoint = torch.load(WEIGHT_PATH, map_location=DEVICE)
    
    # Extract state_dict, stripping 'module.' prefix if necessary
    state_dict = checkpoint['model_state_dict'] if 'model_state_dict' in checkpoint else checkpoint
    clean_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
    model.load_state_dict(clean_state_dict)
    model.eval()

    all_metrics = []

    # --- 4. Main Evaluation Loop ---
    for fname in tqdm(pure_test_list):
        # Filename matching for lesion masks
        match = re.search(r"ISIC_(\d{7})", fname)
        img_id = match.group(1)
        mask_path = os.path.join(MASK_DIR, f"ISIC_{img_id}_lesion.png")

        # Image preprocessing (matching training protocol)
        img = cv2.imread(os.path.join(IMG_DIR, fname))
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img_rgb, (256, 256))
        img_tensor = torch.from_numpy(img_resized).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE) / 255.0
        
        # Mask preprocessing
        mask_gt = cv2.imread(mask_path, 0)
        mask_gt_res = cv2.resize(mask_gt, (256, 256))
        mask_binary = (mask_gt_res > 127).astype(np.uint8)

        # Inference
        with torch.no_grad():
            output = model(img_tensor)
            pred = (output.squeeze().cpu().numpy() > 0.5).astype(np.uint8)

        # Confusion matrix accumulation
        cm = confusion_matrix(mask_binary.flatten(), pred.flatten(), labels=[0, 1])
        all_metrics.append(cm)

    # --- 5. Calculation of Academic Metrics ---
    total_cm = sum(all_metrics)
    tn, fp, fn, tp = total_cm.ravel()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-7)
    iou = tp / (tp + fp + fn + 1e-7)
    acc = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    sen = tp / (tp + fn + 1e-7) # Recall
    spe = tn / (tn + fp + 1e-7)

    print("-" * 30)
    print(f"Baseline Results (Independent Test Set):")
    print(f"Dice (F1):    {dice:.4f}")
    print(f"Jaccard(IoU): {iou:.4f}")
    print(f"Accuracy:     {acc:.4f}")
    print(f"Sensitivity:  {sen:.4f}")
    print(f"Specificity:  {spe:.4f}")
    print("-" * 30)

if __name__ == "__main__":
    run_baseline_pure_test()

Total samples: 493
Testing on 393 independent baseline samples...
SC_Att_Bridge was used


100%|██████████| 393/393 [00:20<00:00, 19.03it/s]

------------------------------
Baseline Results (Independent Test Set):
Dice (F1):    0.7734
Jaccard(IoU): 0.6305
Accuracy:     0.9099
Sensitivity:  0.7094
Specificity:  0.9654
------------------------------
